# 00 — Create BraTS 2021 577-patient manifest

Notebook tự động ghép **BraTS 2021 Task 1** (FLAIR + segmentation) với `train_labels.csv`
(MGMT), kiểm tra đúng cohort **577 bệnh nhân = 301 MGMT=1 + 276 MGMT=0**,
kiểm tra file tồn tại và xuất:

`shared_rao_comparison/brats2021_577_manifest.csv`

Manifest này dùng cho TN11–TN13.


In [ ]:
from __future__ import annotations
from pathlib import Path
import re, json, hashlib
import pandas as pd


## 1. Cấu hình

In [ ]:
TASK1_ROOT = Path("/path/to/BraTS2021_Training_Data")
TRAIN_LABELS_CSV = Path("/path/to/train_labels.csv")

OUTDIR = Path("shared_rao_comparison")
OUTDIR.mkdir(parents=True, exist_ok=True)

MANIFEST_OUT = OUTDIR / "brats2021_577_manifest.csv"
MATCH_REPORT_OUT = OUTDIR / "brats2021_577_matching_report.csv"
MISSING_REPORT_OUT = OUTDIR / "brats2021_577_missing_or_excluded.csv"
SUMMARY_OUT = OUTDIR / "brats2021_577_manifest_summary.json"

EXPECTED_PATIENTS = 577
EXPECTED_MGMT_1 = 301
EXPECTED_MGMT_0 = 276

LABEL_ID_COLUMN = None
LABEL_TARGET_COLUMN = None


## 2. Hàm chuẩn hóa patient ID

In [ ]:
def normalize_numeric_patient_id(value) -> str:
    if pd.isna(value):
        raise ValueError("Patient ID is NaN.")
    text = str(value).strip()
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    m = re.findall(r"(\d{5})", text)
    if m:
        return m[-1]
    if re.fullmatch(r"\d+", text):
        return text.zfill(5)
    raise ValueError(f"Cannot normalize patient ID: {value!r}")

def canonical_patient_id(numeric_id: str) -> str:
    return f"BraTS2021_{numeric_id}"


## 3. Đọc `train_labels.csv` và nhận diện cột

In [ ]:
if not TRAIN_LABELS_CSV.exists():
    raise FileNotFoundError(TRAIN_LABELS_CSV)

labels_raw = pd.read_csv(TRAIN_LABELS_CSV)
print("Columns:", labels_raw.columns.tolist())
display(labels_raw.head())

def detect_column(columns, preferred, candidates):
    if preferred is not None:
        if preferred not in columns:
            raise ValueError(f"Column {preferred!r} not found.")
        return preferred
    lower = {str(c).lower(): c for c in columns}
    for name in candidates:
        if name.lower() in lower:
            return lower[name.lower()]
    for c in columns:
        cl = str(c).lower()
        if any(k.lower() in cl for k in candidates):
            return c
    return None

id_col = detect_column(
    labels_raw.columns, LABEL_ID_COLUMN,
    ["BraTS21ID","BraTS2021ID","patient_id","patient","id"]
)
target_col = detect_column(
    labels_raw.columns, LABEL_TARGET_COLUMN,
    ["MGMT_value","MGMT","mgmt_value","mgmt","label","target"]
)

if id_col is None or target_col is None:
    raise ValueError(
        "Không auto-detect được cột ID/MGMT. "
        "Hãy đặt LABEL_ID_COLUMN và LABEL_TARGET_COLUMN."
    )

print("ID column:", id_col)
print("MGMT column:", target_col)


## 4. Chuẩn hóa nhãn MGMT

In [ ]:
labels = labels_raw[[id_col, target_col]].copy()
labels.columns = ["source_patient_id", "mgmt"]
labels["numeric_id"] = labels["source_patient_id"].map(normalize_numeric_patient_id)
labels["mgmt"] = pd.to_numeric(labels["mgmt"], errors="raise").astype(int)

if not set(labels["mgmt"].unique()).issubset({0,1}):
    raise ValueError(f"MGMT phải là 0/1, found={labels['mgmt'].unique()}")

if labels["numeric_id"].duplicated().any():
    display(labels[labels["numeric_id"].duplicated(keep=False)])
    raise ValueError("Duplicate normalized patient IDs in train_labels.csv")

labels["patient_id"] = labels["numeric_id"].map(canonical_patient_id)

print("Task-2 label rows:", len(labels))
print(labels["mgmt"].value_counts().sort_index())


## 5. Dò folder bệnh nhân Task 1

In [ ]:
if not TASK1_ROOT.exists():
    raise FileNotFoundError(TASK1_ROOT)

def extract_numeric_id(name: str):
    m = re.findall(r"(\d{5})", name)
    return m[-1] if m else None

patient_dirs = {}
for p in TASK1_ROOT.rglob("*"):
    if not p.is_dir():
        continue
    nid = extract_numeric_id(p.name)
    if nid is None:
        continue
    if nid not in patient_dirs or "brats" in p.name.lower():
        patient_dirs[nid] = p

print("Patient-like directories:", len(patient_dirs))


## 6. Dò FLAIR và segmentation

In [ ]:
def is_nifti(p: Path):
    n = p.name.lower()
    return n.endswith(".nii") or n.endswith(".nii.gz")

def choose_unique(candidates, role):
    candidates = sorted(set(candidates))
    if len(candidates) == 0:
        return None, f"missing_{role}"
    if len(candidates) == 1:
        return candidates[0], None
    if role == "flair":
        preferred = [p for p in candidates if re.search(r"_flair\.nii(\.gz)?$", p.name, re.I)]
    else:
        preferred = [p for p in candidates if re.search(r"_seg\.nii(\.gz)?$", p.name, re.I)]
    if len(preferred) == 1:
        return preferred[0], None
    return None, f"ambiguous_{role}:{len(candidates)}"

def find_patient_files(numeric_id: str, patient_dir: Path | None):
    roots = [patient_dir] if patient_dir is not None else [TASK1_ROOT]
    flair_candidates, seg_candidates = [], []
    for root in roots:
        for p in root.rglob("*"):
            if not p.is_file() or not is_nifti(p):
                continue
            if root == TASK1_ROOT and numeric_id not in p.name:
                continue
            lower = p.name.lower()
            if "flair" in lower:
                flair_candidates.append(p)
            if "_seg" in lower or re.search(r"(^|[_\\-.])seg([_\\-.]|$)", lower):
                seg_candidates.append(p)

    flair, fi = choose_unique(flair_candidates, "flair")
    seg, si = choose_unique(seg_candidates, "seg")
    issues = [x for x in [fi, si] if x]
    return flair, seg, ";".join(issues) if issues else None


## 7. Ghép Task 1 + Task 2

In [ ]:
rows = []
for _, row in labels.sort_values("numeric_id").iterrows():
    nid = row["numeric_id"]
    pdir = patient_dirs.get(nid)
    flair, seg, issue = find_patient_files(nid, pdir)

    rows.append({
        "patient_id": row["patient_id"],
        "numeric_id": nid,
        "mgmt": int(row["mgmt"]),
        "task1_patient_dir": str(pdir.resolve()) if pdir else None,
        "flair_path": str(flair.resolve()) if flair else None,
        "seg_path": str(seg.resolve()) if seg else None,
        "status": "ok" if issue is None else issue,
    })

matching = pd.DataFrame(rows).sort_values("numeric_id").reset_index(drop=True)
matching.to_csv(MATCH_REPORT_OUT, index=False)

print("Matching status:")
display(matching["status"].value_counts(dropna=False).to_frame("count"))
display(matching.head())


## 8. Báo cáo missing/excluded

In [ ]:
excluded = matching[matching["status"] != "ok"].copy()
excluded.to_csv(MISSING_REPORT_OUT, index=False)

if len(excluded):
    print("Excluded / unresolved:", len(excluded))
    display(excluded)
else:
    print("No missing or ambiguous cases.")


## 9. Tạo manifest ứng viên

In [ ]:
manifest = (
    matching.loc[
        matching["status"] == "ok",
        ["patient_id","flair_path","seg_path","mgmt"]
    ]
    .copy()
    .sort_values("patient_id")
    .reset_index(drop=True)
)

if manifest["patient_id"].duplicated().any():
    raise ValueError("Duplicate patient_id in final manifest.")

for c in ["flair_path","seg_path"]:
    bad = [p for p in manifest[c] if not Path(p).exists()]
    if bad:
        raise FileNotFoundError(f"{c}: missing file, example={bad[0]}")

print("Candidate patients:", len(manifest))
print(manifest["mgmt"].value_counts().sort_index())
display(manifest.head())


## 10. Khóa cohort 577 / 301 / 276

In [ ]:
n = len(manifest)
n1 = int((manifest["mgmt"] == 1).sum())
n0 = int((manifest["mgmt"] == 0).sum())

errors = []
if n != EXPECTED_PATIENTS:
    errors.append(f"Expected 577 patients, found {n}.")
if n1 != EXPECTED_MGMT_1:
    errors.append(f"Expected MGMT=1 count 301, found {n1}.")
if n0 != EXPECTED_MGMT_0:
    errors.append(f"Expected MGMT=0 count 276, found {n0}.")

if errors:
    print("\n".join(errors))
    raise RuntimeError(
        "Cohort validation failed. Inspect matching/missing reports; "
        "do not force-delete patients to obtain 577."
    )

print("PASS: exact Rao-comparison cohort confirmed.")
print("Patients =", n)
print("MGMT=1  =", n1)
print("MGMT=0  =", n0)


## 11. Xuất manifest

In [ ]:
manifest.to_csv(MANIFEST_OUT, index=False)
print("Saved:", MANIFEST_OUT.resolve())
display(manifest.head(10))


## 12. Final validation

In [ ]:
check = pd.read_csv(MANIFEST_OUT)

assert len(check) == 577
assert check["patient_id"].nunique() == 577
assert int((check["mgmt"] == 1).sum()) == 301
assert int((check["mgmt"] == 0).sum()) == 276
assert check["flair_path"].map(lambda p: Path(p).exists()).all()
assert check["seg_path"].map(lambda p: Path(p).exists()).all()

print("Final manifest validation: PASS")
print(check.shape)
print(check["mgmt"].value_counts().sort_index())


## 13. Ghi audit summary

In [ ]:
def sha256_file(path: Path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

summary = {
    "task1_root": str(TASK1_ROOT.resolve()),
    "train_labels_csv": str(TRAIN_LABELS_CSV.resolve()),
    "train_labels_sha256": sha256_file(TRAIN_LABELS_CSV),
    "detected_id_column": str(id_col),
    "detected_mgmt_column": str(target_col),
    "task2_label_rows": int(len(labels)),
    "final_patients": int(len(manifest)),
    "mgmt_1_methylated": int((manifest["mgmt"] == 1).sum()),
    "mgmt_0_unmethylated": int((manifest["mgmt"] == 0).sum()),
    "excluded_or_unresolved_rows": int(len(excluded)),
    "manifest_path": str(MANIFEST_OUT.resolve()),
    "manifest_sha256": sha256_file(MANIFEST_OUT),
}

SUMMARY_OUT.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(summary, indent=2, ensure_ascii=False))


## 14. Sau khi chạy thành công

Trong **TN11 FIXED**, đặt:

```python
MANIFEST_PATH = Path(
    "shared_rao_comparison/brats2021_577_manifest.csv"
)
```

TN11 sẽ tạo tiếp:

```text
shared_rao_comparison/mgmt_577_fold_assignment_10fold.csv
```

TN12 và TN13 đọc lại đúng file fold đó.
